<a href="https://colab.research.google.com/github/airenas/icefall/blob/master/egs/liepa3/ASR/transcribe_zipformer_vad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Introduction

This colab notebooks shows how to use Python APIs of [sherpa-onnx](https://github.com/k2-fsa/sherpa-onnx) for speech recongition.

# Install sherpa-onnx
And helper tools librosa, silero-vad

In [ ]:
! pip install sherpa-onnx huggingface_hub soundfile silero-vad librosa
print("\nDONE: Installing packages")

# Download model

## Login to HuggingFace

In [ ]:
from huggingface_hub import login

login()
print("\nDONE: login")

## Load model

Model from huggingface_hub here is trained on Lithuanian 450h corpus

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path
import sherpa_onnx

#################################################################
# Download model
#################################################################
def download_files(tag: str) -> (str, str):
    repo_dir = snapshot_download(tag)
    repo_path = Path(repo_dir)
    # return (str(repo_path / "model/model.onnx"), str(repo_path / "model/tokens.txt"))
    return (str(repo_path / "model/model.int8.onnx"), str(repo_path / "model/tokens.txt"))
    

model_tag_hf= "Airenas/liepa3.zipformer-ctc.v01"
model, tokens = download_files(model_tag_hf)
# model, tokens="data/exp06/model.onnx", "data/exp06/tokens.txt"
#################################################################
# Load model into memory
#################################################################

recognizer = sherpa_onnx.OfflineRecognizer.from_zipformer_ctc(
            model=model,
            tokens=tokens,
            debug=True,
            num_threads = 1,
            sample_rate = 16000,
            feature_dim = 80,
        )
print("MODEL LOADED")

from silero_vad import load_silero_vad, get_speech_timestamps

#################################################################
# INIT (do once)
#################################################################
vad_model = load_silero_vad()
print("VAD LOADED: READY to transcribe")


## Select audio

Run and then select wav mono 16kHz prefered

In [ ]:
from IPython.display import display, Audio
import io
import soundfile as sf
import librosa

file_name="audio.wav"
with open(file_name, "rb") as f:
    audio_bytes = f.read()
# from google.colab import files
# uploaded = files.upload()
# audio_bytes = next(iter(uploaded.values()))

# Transcribe audio

In [ ]:
#################################################################
# RECOGNIZE
#################################################################
def recognize(samples, sample_rate, index):
    s = recognizer.create_stream()
    s.accept_waveform(sample_rate, samples)
    recognizer.decode_stream(s)
    print(f"====================================\nChunk {index}")
    display(Audio(samples, rate=sample_rate))
    print(f"====================================\nResult: {s.result.text}")
    # print(f"====================================\nFull result: \n {s.result}")    
#################################################################
# Utils
#################################################################
TARGET_SR=16000
MAX_CHUNK_SEC=60
def make_chunks(speech_timestamps):
    res = []
    start, end = 0, 0
    for ch in speech_timestamps:
        if end - start > MAX_CHUNK_SEC:
            res.append((start, end))
            start = end
        end = ch.get('end', 0)
    if end > start:
       res.append((start, end))
    return res    
#################################################################
# MAIN
#################################################################
samples, sample_rate = sf.read(io.BytesIO(audio_bytes))
print("Sample rate:", sample_rate)
display(Audio(samples, rate=sample_rate))

duration = len(samples) / sample_rate
print("Duration:", duration)

if duration <= MAX_CHUNK_SEC:
    recognize(samples, sample_rate, 0)
else:
    if sample_rate != TARGET_SR:
        print("resample")
        samples = librosa.resample(samples, orig_sr=sample_rate, target_sr=TARGET_SR)
        sample_rate = TARGET_SR

    speech_timestamps = get_speech_timestamps(
        samples,
        vad_model,
        return_seconds=True,
        threshold=0.7,       
        min_speech_duration_ms=500,
        max_speech_duration_s=45,
    )
    # print(speech_timestamps)
    chunks = make_chunks(speech_timestamps)
    # print(chunks)
    for i, (start_sec, end_sec) in enumerate(chunks):
        start = int(start_sec * sample_rate)
        end = int(end_sec * sample_rate)
        recognize(samples[start:end], sample_rate, i)